# Error Analysis — Omission and Addition using Mapped Files

This notebook computes omission and addition error rates for the GEM 2024 ordering and structuring models
using the `.mapped` and `_struct.txt` output files directly, rather than the JSON result files.

Datasets analysed:
- **FA** — Factual
- **FI** — Fictional
- **CFA** — Counterfactual

## File sources

| Role | FA | FI | CFA |
|---|---|---|---|
| Input (source triples) | `factual_input.txt` | `fictional_input.txt` | `counterfactual_input.txt` |
| Ordering output | `factual_ordering.mapped` | `fictional_ordering.mapped` | `counterfactual_ordering.mapped` |
| Structuring output | `factual_struct.txt` | `fictional_struct.txt` | `counterfactual_struct.txt` |

## What is compared

| Analysis | Source | Prediction | What it measures |
|---|---|---|---|
| Ordering vs input | `*_input.txt` | `*_ordering.mapped` | Triples lost or added by the ordering model |
| Structuring vs ordering | `*_ordering.mapped` | `*_struct.txt` | Triples lost or added by structuring (structuring-specific errors only) |
| Structuring vs input | `*_input.txt` | `*_struct.txt` | Cumulative pipeline loss across both stages |

## Matching logic
Predicates are matched **greedily** (first match wins) and **case-sensitively**.
A wrong-capitalisation prediction (e.g. `ICAOLocationIdentifier` vs `icaoLocationIdentifier`) counts as one omission + one addition.

In [1]:
from pathlib import Path
import re
import csv
from collections import defaultdict

try:
    import pandas as pd
except ImportError:
    pd = None

BASE_DIR        = Path('/Users/chinonsoosuji/Python_projects/PHD PROJECTS/GEM2024_ST')
ORDERING_DIR    = BASE_DIR / 'results' / 'ordering'
STRUCTURING_DIR = BASE_DIR / 'results' / 'structuring'
INPUT_DIR       = BASE_DIR / 'Evaluation' / 'input'
OUTPUT_DIR      = BASE_DIR / 'results' / 'error_analysis_mapped'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# File paths for each dataset
DATASETS = {
    'FA':  {
        'input':        INPUT_DIR       / 'factual_input.txt',
        'ordering':     ORDERING_DIR    / 'factual_ordering.mapped',
        'structuring':  STRUCTURING_DIR / 'factual_struct.txt',
    },
    'FI':  {
        'input':        INPUT_DIR       / 'fictional_input.txt',
        'ordering':     ORDERING_DIR    / 'fictional_ordering.mapped',
        'structuring':  STRUCTURING_DIR / 'fictional_struct.txt',
    },
    'CFA': {
        'input':        INPUT_DIR       / 'counterfactual_input.txt',
        'ordering':     ORDERING_DIR    / 'counterfactual_ordering.mapped',
        'structuring':  STRUCTURING_DIR / 'counterfactual_struct.txt',
    },
}

print('Paths ready.')
for ds, paths in DATASETS.items():
    for role, path in paths.items():
        exists = '✓' if path.exists() else '✗ MISSING'
        print(f'  {ds:4s} {role:12s} {exists}  {path.name}')

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


Paths ready.
  FA   input        ✓  factual_input.txt
  FA   ordering     ✓  factual_ordering.mapped
  FA   structuring  ✓  factual_struct.txt
  FI   input        ✓  fictional_input.txt
  FI   ordering     ✓  fictional_ordering.mapped
  FI   structuring  ✓  fictional_struct.txt
  CFA  input        ✓  counterfactual_input.txt
  CFA  ordering     ✓  counterfactual_ordering.mapped
  CFA  structuring  ✓  counterfactual_struct.txt


## Parsing functions

In [2]:
TRIPLE_RE = re.compile(r'\[TRIPLE\](.*?)\[/TRIPLE\]', flags=re.IGNORECASE | re.DOTALL)


def parse_input_txt_line(line):
    """
    Parse one line from *_input.txt.

    Format: 'subject  predicate  object, subject  predicate  object, ...'
    - Triples are separated by ', ' (comma + space).
    - Within each triple, subject / predicate / object are separated by
      two spaces ('  ').

    Returns a list of triple dicts with keys:
        source_triple_index, subject, predicate, object, raw
    """
    triples = []
    for i, triple_str in enumerate(line.strip().split(', ')):
        parts = [p for p in triple_str.split('  ') if p.strip()]
        if len(parts) >= 2:
            triples.append({
                'source_triple_index': i,
                'subject':   parts[0].strip(),
                'predicate': parts[1].strip(),
                'object':    parts[2].strip() if len(parts) > 2 else '',
                'raw':       triple_str.strip(),
            })
    return triples


def parse_mapped_line(line):
    """
    Parse one line from *_ordering.mapped or *_struct.txt.

    Both formats use [TRIPLE] subject predicate object [/TRIPLE] tags.
    The structuring files also wrap triples in [SNT]...[/SNT] sentence groups,
    but we ignore [SNT] tags here and extract all [TRIPLE] tags regardless.

    Returns a list of triple dicts with keys:
        source_triple_index, subject, predicate, object, raw
    """
    triples = []
    for i, match in enumerate(TRIPLE_RE.finditer(line)):
        raw = re.sub(r'\s+', ' ', match.group(1)).strip()
        parts = raw.split(maxsplit=2)
        if len(parts) >= 2:
            triples.append({
                'source_triple_index': i,
                'subject':   parts[0],
                'predicate': parts[1],
                'object':    parts[2] if len(parts) == 3 else '',
                'raw':       raw,
            })
    return triples


def read_lines(path):
    """Read a file and return non-empty lines."""
    with open(path, encoding='utf-8') as f:
        return [line for line in f.read().splitlines() if line.strip()]


print('Parsing functions ready.')

Parsing functions ready.


## Matching and comparison

In [3]:
def match_predicates(source_triples, predicted_triples):
    """
    Greedy, case-sensitive one-to-one matching of predicted predicates to source triples.

    For each predicted triple (in order), we look for the first unmatched source
    triple whose predicate field is an exact string match. Once matched, neither
    the source triple nor the predicted triple can be matched again.

    Returns:
        omitted — source triples that had no matching prediction (dropped triples)
        added   — predicted triples that had no matching source triple (invented triples)
        n_matched — number of successfully matched pairs
    """
    matched_source = set()
    matched_pred   = set()

    for pred_idx, pred_triple in enumerate(predicted_triples):
        for src_idx, src_triple in enumerate(source_triples):
            if src_idx in matched_source:
                continue
            if pred_triple['predicate'] == src_triple['predicate']:
                matched_source.add(src_idx)
                matched_pred.add(pred_idx)
                break

    omitted = [t for i, t in enumerate(source_triples)    if i not in matched_source]
    added   = [t for i, t in enumerate(predicted_triples) if i not in matched_pred]
    return omitted, added, len(matched_source)


print('Matching function ready.')

Matching function ready.


## Run error analysis for FA, FI, CFA

In [4]:
# Each row in detail_rows holds per-example counts for one (dataset, analysis_type) pair.
detail_rows = []

for dataset, paths in DATASETS.items():
    input_lines       = read_lines(paths['input'])
    ordering_lines    = read_lines(paths['ordering'])
    structuring_lines = read_lines(paths['structuring'])

    for idx, (inp, ord_out, struct_out) in enumerate(
        zip(input_lines, ordering_lines, structuring_lines)
    ):
        src_triples   = parse_input_txt_line(inp)       # original WebNLG-17 triples
        ord_triples   = parse_mapped_line(ord_out)      # ordering model output
        struct_triples = parse_mapped_line(struct_out)  # structuring model output

        # --- Analysis 1: Ordering vs original input ---
        # Measures triples lost or invented by the ordering stage.
        omitted_ord, added_ord, matched_ord = match_predicates(src_triples, ord_triples)
        detail_rows.append({
            'dataset':           dataset,
            'analysis':          'ordering_vs_input',
            'example_id':        idx,
            'n_source':          len(src_triples),
            'n_predicted':       len(ord_triples),
            'n_matched':         matched_ord,
            'n_omitted':         len(omitted_ord),
            'n_added':           len(added_ord),
            'omitted_predicates': ' | '.join(t['predicate'] for t in omitted_ord),
            'added_predicates':  ' | '.join(t['predicate'] for t in added_ord),
        })

        # --- Analysis 2: Structuring vs ordering output ---
        # Measures triples lost or invented by structuring ALONE
        # (the ordering output is the structuring model's direct input).
        omitted_st, added_st, matched_st = match_predicates(ord_triples, struct_triples)
        detail_rows.append({
            'dataset':           dataset,
            'analysis':          'structuring_vs_ordering',
            'example_id':        idx,
            'n_source':          len(ord_triples),
            'n_predicted':       len(struct_triples),
            'n_matched':         matched_st,
            'n_omitted':         len(omitted_st),
            'n_added':           len(added_st),
            'omitted_predicates': ' | '.join(t['predicate'] for t in omitted_st),
            'added_predicates':  ' | '.join(t['predicate'] for t in added_st),
        })

        # --- Analysis 3: Structuring vs original input ---
        # Cumulative pipeline loss: triples lost across BOTH ordering and structuring.
        # Do NOT attribute this to structuring alone.
        omitted_cum, added_cum, matched_cum = match_predicates(src_triples, struct_triples)
        detail_rows.append({
            'dataset':           dataset,
            'analysis':          'structuring_vs_input',
            'example_id':        idx,
            'n_source':          len(src_triples),
            'n_predicted':       len(struct_triples),
            'n_matched':         matched_cum,
            'n_omitted':         len(omitted_cum),
            'n_added':           len(added_cum),
            'omitted_predicates': ' | '.join(t['predicate'] for t in omitted_cum),
            'added_predicates':  ' | '.join(t['predicate'] for t in added_cum),
        })

print(f'Total detail rows: {len(detail_rows)}')
print(f'  = {len(DATASETS)} datasets × 1779 examples × 3 analysis types')

Total detail rows: 16011
  = 3 datasets × 1779 examples × 3 analysis types


## Summary statistics

In [5]:
def summarise(rows, group_keys):
    """
    Aggregate detail rows into summary statistics, grouped by group_keys.

    omitted_pct = total_omitted / total_source * 100
    added_pct   = total_added   / total_source * 100
    Both denominators use the SOURCE triple count so the two rates are directly comparable.
    """
    groups = defaultdict(list)
    for row in rows:
        key = tuple(row[k] for k in group_keys)
        groups[key].append(row)

    summary = []
    for key_vals, grp in sorted(groups.items()):
        n_ex      = len(grp)
        n_src     = sum(r['n_source']   for r in grp)
        n_pred    = sum(r['n_predicted']for r in grp)
        n_omit    = sum(r['n_omitted']  for r in grp)
        n_add     = sum(r['n_added']    for r in grp)
        w_omit    = sum(r['n_omitted'] > 0 for r in grp)
        w_add     = sum(r['n_added']   > 0 for r in grp)

        summary.append({
            **dict(zip(group_keys, key_vals)),
            'n_examples':          n_ex,
            'total_source':        n_src,
            'total_predicted':     n_pred,
            'total_omitted':       n_omit,
            'omitted_pct':         round(n_omit / n_src * 100, 2) if n_src else 0.0,
            'total_added':         n_add,
            'added_pct':           round(n_add  / n_src * 100, 2) if n_src else 0.0,
            'examples_w_omissions':w_omit,
            'omissions_pct':       round(w_omit / n_ex  * 100, 2) if n_ex  else 0.0,
            'examples_w_additions':w_add,
            'additions_pct':       round(w_add  / n_ex  * 100, 2) if n_ex  else 0.0,
        })
    return summary


# Per-dataset summary (FA / FI / CFA × 3 analysis types = 9 rows)
dataset_summary = summarise(detail_rows, ['dataset', 'analysis'])

# Overall summary across all datasets (3 analysis types = 3 rows)
overall_summary = summarise(detail_rows, ['analysis'])

print(f'Dataset summary rows: {len(dataset_summary)}')
print(f'Overall summary rows: {len(overall_summary)}')

Dataset summary rows: 9
Overall summary rows: 3


In [6]:
# Both come from the summarise function in the notebook:

# n_examples — the number of individual examples (triple sets) in that group. Since each dataset has 1,779 examples, n_examples = 1779 for a single dataset row, and 5,337 for the TOTAL row (1,779 × 3 datasets).

# total_source — the total number of source triples across all examples in that group. This is the denominator used to calculate omitted_pct and added_pct. What it counts depends on the analysis type:

# Analysis	What total_source counts
# ordering_vs_input	Triples in the original *_input.txt — what the model was supposed to order
# structuring_vs_ordering	Triples in *_ordering.mapped — what the structuring model actually received as input
# structuring_vs_input	Triples in the original *_input.txt — used for cumulative pipeline loss
# So total_source is essentially "how many triples should have appeared in the output?" and every omission/addition percentage is calculated relative to it. For example, FA ordering: total_source = 5,639 means there were 5,639 source triples across all 1,779 factual examples, and omitted_pct = 592 / 5639 × 100 = 10.50%.

# examples_w_omissions is the count of individual examples (triple sets) where at least one triple was omitted.

# For instance, in the FA ordering analysis:

# There are 1,779 examples total
# 384 of those examples had at least one triple that the model failed to include
# The remaining 1,395 examples (1,779 − 384) had zero omissions — the model included every triple correctly
# So it answers the question: "How widespread is the omission problem?" — not how many triples were dropped in total, but how many examples were affected by at least one dropped triple.

# The difference between examples_w_omissions and total_omitted:

# Metric	What it tells you
# total_omitted	Total number of triple slots dropped across the whole dataset
# examples_w_omissions	Number of examples that experienced at least one omission
# omissions_pct	Percentage of examples affected (384 / 1779 × 100 = 21.59%)
# A practical example: if one difficult example had 5 triples all omitted, that contributes 5 to total_omitted but only 1 to examples_w_omissions. The two metrics together tell you both the severity (how many triples lost) and the breadth (how many examples affected) of the omission problem.

## Table 1 — Ordering errors by dataset (vs original input)

In [7]:
def build_table(summary_rows, analysis_filter, title, include_total=True):
    """
    Display a formatted table for one analysis type.
    Adds a TOTAL row computed from raw counts (not by averaging percentages).
    """
    subset = [r for r in summary_rows if r['analysis'] == analysis_filter]

    if include_total:
        n_src  = sum(r['total_source']  for r in subset)
        n_omit = sum(r['total_omitted'] for r in subset)
        n_add  = sum(r['total_added']   for r in subset)
        n_ex   = sum(r['n_examples']    for r in subset)
        w_omit = sum(r['examples_w_omissions'] for r in subset)
        w_add  = sum(r['examples_w_additions'] for r in subset)
        subset = subset + [{
            'dataset':              'TOTAL',
            'analysis':             analysis_filter,
            'n_examples':           n_ex,
            'total_source':         n_src,
            'total_omitted':        n_omit,
            'omitted_pct':          round(n_omit / n_src * 100, 2) if n_src else 0.0,
            'total_added':          n_add,
            'added_pct':            round(n_add  / n_src * 100, 2) if n_src else 0.0,
            'examples_w_omissions': w_omit,
            'omissions_pct':        round(w_omit / n_ex  * 100, 2) if n_ex  else 0.0,
            'examples_w_additions': w_add,
            'additions_pct':        round(w_add  / n_ex  * 100, 2) if n_ex  else 0.0,
        }]

    cols = [
        'dataset', 'n_examples', 'total_source',
        'total_omitted', 'omitted_pct',
        'total_added',   'added_pct',
        'examples_w_omissions', 'omissions_pct',
        'examples_w_additions', 'additions_pct',
    ]
    rename = {
        'dataset':               'Dataset',
        'n_examples':            'Examples',
        'total_source':          'Source Triples',
        'total_omitted':         'Omitted',
        'omitted_pct':           'Omission %',
        'total_added':           'Added',
        'added_pct':             'Addition %',
        'examples_w_omissions':  'w/ Omissions',
        'omissions_pct':         'Omissions %',
        'examples_w_additions':  'w/ Additions',
        'additions_pct':         'Additions %',
    }

    df = pd.DataFrame(subset)[cols].rename(columns=rename)
    print(f'\n{title}')
    print('=' * len(title))
    print(df.to_string(index=False))
    return df


t1 = build_table(
    dataset_summary,
    'ordering_vs_input',
    'Table 1: Ordering errors by dataset — triples lost or added vs original input',
)


Table 1: Ordering errors by dataset — triples lost or added vs original input
Dataset  Examples  Source Triples  Omitted  Omission %  Added  Addition %  w/ Omissions  Omissions %  w/ Additions  Additions %
    CFA      1779            5639      529        9.38      0         0.0           352        19.79             0          0.0
     FA      1779            5639      592       10.50      0         0.0           384        21.59             0          0.0
     FI      1779            5639      513        9.10      0         0.0           333        18.72             0          0.0
  TOTAL      5337           16917     1634        9.66      0         0.0          1069        20.03             0          0.0


## Table 2 — Structuring-specific errors by dataset (vs ordering output)

Source triples here = ordering output triples (what structuring received as input).
These numbers isolate errors made by the structuring model alone.

In [8]:
t2 = build_table(
    dataset_summary,
    'structuring_vs_ordering',
    'Table 2: Structuring-specific errors by dataset — triples lost or added vs ordering output',
)


Table 2: Structuring-specific errors by dataset — triples lost or added vs ordering output
Dataset  Examples  Source Triples  Omitted  Omission %  Added  Addition %  w/ Omissions  Omissions %  w/ Additions  Additions %
    CFA      1779            5110        4        0.08      0         0.0             4         0.22             0          0.0
     FA      1779            5047        3        0.06      0         0.0             3         0.17             0          0.0
     FI      1779            5126        4        0.08      0         0.0             4         0.22             0          0.0
  TOTAL      5337           15283       11        0.07      0         0.0            11         0.21             0          0.0


In [9]:
# 15,283 is the total number of triples across all three datasets that the ordering model passed to structuring — it is the total_source for the TOTAL row in the structuring vs ordering analysis.

# Here is where it comes from:

# Dataset	Ordering output triples	(= source triples − omitted by ordering)
# FA	5,047	5,639 − 592
# FI	5,126	5,639 − 513
# CFA	5,110	5,639 − 529
# TOTAL	15,283	16,917 − 1,634
# So 15,283 ≠ 16,917 because the ordering model already dropped 1,634 triples before its output was passed to structuring. Those dropped triples simply never reached the structuring model.

# This is why the structuring_vs_ordering analysis uses 15,283 as its denominator instead of 16,917 — it is asking: "out of the triples structuring actually received, how many did it drop?" That is the correct denominator for isolating structuring-specific errors. Using 16,917 would be wrong because it would include triples that structuring never even saw.

## Table 3 — Cumulative pipeline loss by dataset (vs original input)

Source triples here = original WebNLG-17 input.
These numbers reflect combined loss from both ordering and structuring.
Do NOT attribute this to structuring alone.

In [10]:
t3 = build_table(
    dataset_summary,
    'structuring_vs_input',
    'Table 3: Cumulative pipeline loss by dataset — triples lost or added vs original input',
)


Table 3: Cumulative pipeline loss by dataset — triples lost or added vs original input
Dataset  Examples  Source Triples  Omitted  Omission %  Added  Addition %  w/ Omissions  Omissions %  w/ Additions  Additions %
    CFA      1779            5639      533        9.45      0         0.0           355        19.96             0          0.0
     FA      1779            5639      595       10.55      0         0.0           385        21.64             0          0.0
     FI      1779            5639      517        9.17      0         0.0           335        18.83             0          0.0
  TOTAL      5337           16917     1645        9.72      0         0.0          1075        20.14             0          0.0


## Omission distribution — how many triples were dropped per example

These tables show how many examples had exactly 1, 2, 3, … triples omitted.
This reveals whether omissions are spread across many examples (lots of 1-triple drops)
or concentrated in a few difficult examples (several large drops).

- **Column 0** — examples with zero omissions (model reproduced every triple correctly)
- **Columns 1–7** — examples where exactly N triples were omitted
- **Total affected** — all examples with at least one omission (= sum of columns 1–7)

In [11]:
from collections import Counter

MAX_SHOW = 7  # show distribution columns for 0 through 7 omissions


def omission_distribution(detail_rows, dataset_filter, analysis_filter):
    """
    For a given dataset and analysis type, count how many examples had
    exactly 0, 1, 2, ... MAX_SHOW omitted triples.

    Returns a dict: {0: count, 1: count, ..., MAX_SHOW: count, 'total_affected': count}
    'total_affected' = examples with at least 1 omission (columns 1 through MAX_SHOW).
    """
    subset = [
        r for r in detail_rows
        if r['dataset'] == dataset_filter and r['analysis'] == analysis_filter
    ]
    counts = Counter(r['n_omitted'] for r in subset)
    dist = {n: counts.get(n, 0) for n in range(MAX_SHOW + 1)}
    dist['total_affected'] = sum(dist[n] for n in range(1, MAX_SHOW + 1))
    return dist


def print_distribution_table(analysis_filter, title):
    """
    Build and print a distribution table for one analysis type.
    Rows = datasets (FA, FI, CFA) + TOTAL.
    Columns = number of omissions (0 through MAX_SHOW) + total_affected.
    """
    table_rows = []
    for ds in ['FA', 'FI', 'CFA']:
        dist = omission_distribution(detail_rows, ds, analysis_filter)
        row = {'Dataset': ds, **{str(n): dist[n] for n in range(MAX_SHOW + 1)},
               'Total affected': dist['total_affected']}
        table_rows.append(row)

    # TOTAL row: sum each column across all datasets
    total_row = {'Dataset': 'TOTAL'}
    for n in range(MAX_SHOW + 1):
        total_row[str(n)] = sum(r[str(n)] for r in table_rows)
    total_row['Total affected'] = sum(r['Total affected'] for r in table_rows)
    table_rows.append(total_row)

    df = pd.DataFrame(table_rows)
    col_order = ['Dataset'] + [str(n) for n in range(MAX_SHOW + 1)] + ['Total affected']
    df = df[col_order]

    # Rename numeric columns to make the header clearer
    df = df.rename(columns={str(n): f'{n} omitted' for n in range(MAX_SHOW + 1)})

    print(f'\n{title}')
    print('=' * len(title))
    print(df.to_string(index=False))
    return df


dist1 = print_distribution_table(
    'ordering_vs_input',
    'Distribution 1: Ordering — omitted triples per example (vs original input)',
)

dist2 = print_distribution_table(
    'structuring_vs_ordering',
    'Distribution 2: Structuring — omitted triples per example (vs ordering output)',
)

dist3 = print_distribution_table(
    'structuring_vs_input',
    'Distribution 3: Structuring — omitted triples per example (vs original input, cumulative)',
)


Distribution 1: Ordering — omitted triples per example (vs original input)
Dataset  0 omitted  1 omitted  2 omitted  3 omitted  4 omitted  5 omitted  6 omitted  7 omitted  Total affected
     FA       1395        244         93         31         11          5          0          0             384
     FI       1446        212         83         20         15          3          0          0             333
    CFA       1427        227         86         28          9          2          0          0             352
  TOTAL       4268        683        262         79         35         10          0          0            1069

Distribution 2: Structuring — omitted triples per example (vs ordering output)
Dataset  0 omitted  1 omitted  2 omitted  3 omitted  4 omitted  5 omitted  6 omitted  7 omitted  Total affected
     FA       1776          3          0          0          0          0          0          0               3
     FI       1775          4          0          0         

## Omission rate by source triple count

This section shows how omission rates change as the number of source triples in an example increases.
Each row groups all examples (across FA, FI, CFA) that had exactly N source triples.

**Hypothesis**: examples with more triples are harder to process, leading to higher omission rates.

Columns:
- **n_source** — number of triples in the input for that example
- **n_examples** — total examples with that triple count
- **w_omissions** — examples where at least one triple was omitted
- **om_rate_%** — share of *examples* with at least one omission (`w_omissions / n_examples × 100`)
- **total_omitted** — total triple slots dropped across all examples in this group
- **avg_omitted** — average omissions per example (`total_omitted / n_examples`)
- **omitted_pct** — share of *source triples* dropped (`total_omitted / total_source × 100`)

In [12]:
def omission_by_source_size(detail_rows, analysis_filter, datasets=('FA', 'FI', 'CFA')):
    """
    Group examples by their n_source (triple-set size) and compute omission statistics.

    Aggregates across all specified datasets so each row represents ALL examples
    of a given input size (e.g. all examples that had exactly 3 source triples).

    Returns a list of dicts sorted by n_source, plus a TOTAL row.
    """
    from collections import defaultdict

    groups = defaultdict(list)
    for row in detail_rows:
        if row['analysis'] == analysis_filter and row['dataset'] in datasets:
            groups[row['n_source']].append(row)

    table_rows = []
    for n_src in sorted(groups):
        grp = groups[n_src]
        n_ex      = len(grp)
        total_src = sum(r['n_source']  for r in grp)   # = n_src * n_ex
        total_om  = sum(r['n_omitted'] for r in grp)
        w_om      = sum(r['n_omitted'] > 0 for r in grp)
        table_rows.append({
            'n_source':      n_src,
            'n_examples':    n_ex,
            'w_omissions':   w_om,
            'om_rate_%':     round(w_om    / n_ex      * 100, 2) if n_ex  else 0.0,
            'total_omitted': total_om,
            'avg_omitted':   round(total_om / n_ex,            2) if n_ex  else 0.0,
            'omitted_pct':   round(total_om / total_src * 100, 2) if total_src else 0.0,
        })

    # TOTAL row
    all_grp = [r for r in detail_rows
               if r['analysis'] == analysis_filter and r['dataset'] in datasets]
    n_ex      = len(all_grp)
    total_src = sum(r['n_source']  for r in all_grp)
    total_om  = sum(r['n_omitted'] for r in all_grp)
    w_om      = sum(r['n_omitted'] > 0 for r in all_grp)
    table_rows.append({
        'n_source':      'TOTAL',
        'n_examples':    n_ex,
        'w_omissions':   w_om,
        'om_rate_%':     round(w_om    / n_ex      * 100, 2) if n_ex  else 0.0,
        'total_omitted': total_om,
        'avg_omitted':   round(total_om / n_ex,            2) if n_ex  else 0.0,
        'omitted_pct':   round(total_om / total_src * 100, 2) if total_src else 0.0,
    })
    return table_rows


def print_source_size_table(analysis_filter, title):
    rows = omission_by_source_size(detail_rows, analysis_filter)
    df = pd.DataFrame(rows)
    print(f'\n{title}')
    print('=' * len(title))
    print(df.to_string(index=False))
    return df


sz1 = print_source_size_table(
    'ordering_vs_input',
    'Ordering omission rate by source triple count (all datasets combined)',
)

sz3 = print_source_size_table(
    'structuring_vs_input',
    'Cumulative omission rate by source triple count (all datasets combined)',
)


Ordering omission rate by source triple count (all datasets combined)
n_source  n_examples  w_omissions  om_rate_%  total_omitted  avg_omitted  omitted_pct
       1        1107            0       0.00              0         0.00         0.00
       2        1047           74       7.07             74         0.07         3.53
       3        1050          200      19.05            266         0.25         8.44
       4         915          262      28.63            371         0.41        10.14
       5         639          250      39.12            406         0.64        12.71
       6         342          148      43.27            272         0.80        13.26
       7         237          135      56.96            245         1.03        14.77
   TOTAL        5337         1069      20.03           1634         0.31         9.66

Cumulative omission rate by source triple count (all datasets combined)
n_source  n_examples  w_omissions  om_rate_%  total_omitted  avg_omitted  omitted_

In [13]:
def omission_by_source_size_per_dataset(detail_rows, analysis_filter):
    """
    Build a wide comparison table: one row per n_source, columns split by dataset.

    For each (n_source, dataset) pair we report:
      n_ex        — number of examples with that triple count
      w_om        — examples with at least one omission
      om_rate_%   — w_om / n_ex * 100
      omitted_pct — total_omitted / total_source * 100
    """
    from collections import defaultdict

    # Collect per-dataset stats keyed by n_source
    ds_stats = {}   # ds_stats[dataset][n_source] = {n_ex, w_om, total_om, total_src}
    all_n_sources = set()

    for ds in ['FA', 'FI', 'CFA']:
        groups = defaultdict(list)
        for row in detail_rows:
            if row['analysis'] == analysis_filter and row['dataset'] == ds:
                groups[row['n_source']].append(row)
        ds_stats[ds] = {}
        for n_src, grp in groups.items():
            all_n_sources.add(n_src)
            ds_stats[ds][n_src] = {
                'n_ex':      len(grp),
                'w_om':      sum(r['n_omitted'] > 0 for r in grp),
                'total_om':  sum(r['n_omitted']     for r in grp),
                'total_src': sum(r['n_source']       for r in grp),
            }

    # Build wide rows
    table_rows = []
    for n_src in sorted(all_n_sources):
        row = {'n_source': n_src}
        for ds in ['FA', 'FI', 'CFA']:
            s = ds_stats[ds].get(n_src, {'n_ex': 0, 'w_om': 0, 'total_om': 0, 'total_src': 0})
            row[f'{ds}_n_ex']       = s['n_ex']
            row[f'{ds}_w_om']       = s['w_om']
            row[f'{ds}_om_rate%']   = round(s['w_om']     / s['n_ex']      * 100, 1) if s['n_ex']      else 0.0
            row[f'{ds}_omit_pct%']  = round(s['total_om'] / s['total_src'] * 100, 1) if s['total_src'] else 0.0
        table_rows.append(row)

    # TOTAL row
    total_row = {'n_source': 'TOTAL'}
    for ds in ['FA', 'FI', 'CFA']:
        all_grp = [r for r in detail_rows
                   if r['analysis'] == analysis_filter and r['dataset'] == ds]
        n_ex      = len(all_grp)
        total_src = sum(r['n_source']  for r in all_grp)
        total_om  = sum(r['n_omitted'] for r in all_grp)
        w_om      = sum(r['n_omitted'] > 0 for r in all_grp)
        total_row[f'{ds}_n_ex']      = n_ex
        total_row[f'{ds}_w_om']      = w_om
        total_row[f'{ds}_om_rate%']  = round(w_om     / n_ex      * 100, 1) if n_ex      else 0.0
        total_row[f'{ds}_omit_pct%'] = round(total_om / total_src * 100, 1) if total_src else 0.0
    table_rows.append(total_row)
    return table_rows


def print_per_dataset_size_table(analysis_filter, title):
    rows = omission_by_source_size_per_dataset(detail_rows, analysis_filter)
    df = pd.DataFrame(rows)

    # Group columns for clarity
    col_order = ['n_source']
    for ds in ['FA', 'FI', 'CFA']:
        col_order += [f'{ds}_n_ex', f'{ds}_w_om', f'{ds}_om_rate%', f'{ds}_omit_pct%']
    df = df[col_order]

    print(f'\n{title}')
    print('=' * len(title))
    print(df.to_string(index=False))
    return df


sz1_ds = print_per_dataset_size_table(
    'ordering_vs_input',
    'Ordering omission rate by source triple count — FA vs FI vs CFA',
)

print()

sz3_ds = print_per_dataset_size_table(
    'structuring_vs_input',
    'Cumulative omission rate by source triple count — FA vs FI vs CFA',
)


Ordering omission rate by source triple count — FA vs FI vs CFA
n_source  FA_n_ex  FA_w_om  FA_om_rate%  FA_omit_pct%  FI_n_ex  FI_w_om  FI_om_rate%  FI_omit_pct%  CFA_n_ex  CFA_w_om  CFA_om_rate%  CFA_omit_pct%
       1      369        0          0.0           0.0      369        0          0.0           0.0       369         0           0.0            0.0
       2      349       31          8.9           4.4      349       21          6.0           3.0       349        22           6.3            3.2
       3      350       72         20.6           9.1      350       61         17.4           7.6       350        67          19.1            8.6
       4      305       95         31.1          10.7      305       81         26.6           9.7       305        86          28.2           10.0
       5      213       88         41.3          13.7      213       76         35.7          12.0       213        86          40.4           12.4
       6      114       53         46.5        

## Save results

In [14]:
def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        return
    fieldnames = list(rows[0].keys())
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


details_path         = OUTPUT_DIR / 'error_analysis_details.csv'
dataset_summary_path = OUTPUT_DIR / 'error_analysis_dataset_summary.csv'
overall_summary_path = OUTPUT_DIR / 'error_analysis_overall_summary.csv'
dist_path            = OUTPUT_DIR / 'error_analysis_omission_distribution.csv'
sz_ordering_path     = OUTPUT_DIR / 'omission_by_source_size_ordering.csv'
sz_cumulative_path   = OUTPUT_DIR / 'omission_by_source_size_cumulative.csv'
sz_ordering_ds_path  = OUTPUT_DIR / 'omission_by_source_size_ordering_per_dataset.csv'
sz_cumul_ds_path     = OUTPUT_DIR / 'omission_by_source_size_cumulative_per_dataset.csv'

write_csv(details_path, detail_rows)
write_csv(dataset_summary_path, dataset_summary)
write_csv(overall_summary_path, overall_summary)

# Distribution tables (one CSV with analysis column)
dist_rows = []
for analysis_filter, label in [
    ('ordering_vs_input',       'ordering_vs_input'),
    ('structuring_vs_ordering', 'structuring_vs_ordering'),
    ('structuring_vs_input',    'structuring_vs_input'),
]:
    for ds in ['FA', 'FI', 'CFA']:
        dist = omission_distribution(detail_rows, ds, analysis_filter)
        row = {'analysis': label, 'dataset': ds}
        for n in range(MAX_SHOW + 1):
            row[f'omitted_{n}'] = dist[n]
        row['total_affected'] = dist['total_affected']
        dist_rows.append(row)
write_csv(dist_path, dist_rows)

# Source-size breakdown — combined and per-dataset
write_csv(sz_ordering_path,   omission_by_source_size(detail_rows, 'ordering_vs_input'))
write_csv(sz_cumulative_path, omission_by_source_size(detail_rows, 'structuring_vs_input'))
write_csv(sz_ordering_ds_path,  omission_by_source_size_per_dataset(detail_rows, 'ordering_vs_input'))
write_csv(sz_cumul_ds_path,     omission_by_source_size_per_dataset(detail_rows, 'structuring_vs_input'))

print('Saved:')
for p in [details_path, dataset_summary_path, overall_summary_path,
          dist_path, sz_ordering_path, sz_cumulative_path,
          sz_ordering_ds_path, sz_cumul_ds_path]:
    print(f'  - {p.relative_to(BASE_DIR)}')

Saved:
  - results/error_analysis_mapped/error_analysis_details.csv
  - results/error_analysis_mapped/error_analysis_dataset_summary.csv
  - results/error_analysis_mapped/error_analysis_overall_summary.csv
  - results/error_analysis_mapped/error_analysis_omission_distribution.csv
  - results/error_analysis_mapped/omission_by_source_size_ordering.csv
  - results/error_analysis_mapped/omission_by_source_size_cumulative.csv
  - results/error_analysis_mapped/omission_by_source_size_ordering_per_dataset.csv
  - results/error_analysis_mapped/omission_by_source_size_cumulative_per_dataset.csv
